### Question-2

In [ ]:
pip install torch torchvision matplotlib scikit-learn pandas

### Imports

In [ ]:
import os
import copy
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim

from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms, models

from sklearn.metrics import confusion_matrix, f1_score

### Set Seed and device

In [ ]:
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

In [ ]:
class CustomCNN(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()

        self.features = nn.Sequential(
            # Block 1
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),   # conv1
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),

            nn.Conv2d(32, 32, kernel_size=3, stride=1, padding=1),  # conv2
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 2
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),  # conv3
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),

            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1),  # conv4
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),

            # Block 3
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1), # conv5
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),

            nn.Conv2d(128, 128, kernel_size=3, stride=1, padding=1),# conv6
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(2),
            nn.Dropout(0.25),
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 4 * 4, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [ ]:
def get_transforms(model_name):
    model_name = model_name.lower()

    if model_name == "customcnn":
        train_transform = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.5071, 0.4867, 0.4408),
                                 (0.2675, 0.2565, 0.2761))
        ])

        test_transform = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.5071, 0.4867, 0.4408),
                                 (0.2675, 0.2565, 0.2761))
        ])
    else:
        train_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406),
                                 (0.229, 0.224, 0.225))
        ])

        test_transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize((0.485, 0.456, 0.406),
                                 (0.229, 0.224, 0.225))
        ])

    return train_transform, test_transform

In [ ]:
def get_dataloaders(model_name, batch_size=128, val_ratio=0.1):
    train_transform, test_transform = get_transforms(model_name)

    full_train_dataset = datasets.CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=train_transform
    )

    test_dataset = datasets.CIFAR100(
        root="./data",
        train=False,
        download=True,
        transform=test_transform
    )

    val_size = int(len(full_train_dataset) * val_ratio)
    train_size = len(full_train_dataset) - val_size

    train_dataset, val_dataset_temp = random_split(
        full_train_dataset,
        [train_size, val_size],
        generator=torch.Generator().manual_seed(42)
    )

    # Validation without augmentation
    base_train_for_val = datasets.CIFAR100(
        root="./data",
        train=True,
        download=True,
        transform=test_transform
    )
    val_dataset = torch.utils.data.Subset(base_train_for_val, val_dataset_temp.indices)

    train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=2)
    val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=2)
    test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=2)

    return train_loader, val_loader, test_loader

In [ ]:
def get_model(model_name, num_classes=100, pretrained=True):
    model_name = model_name.lower()

    if model_name == "customcnn":
        return CustomCNN(num_classes=num_classes)

    elif model_name == "resnet50":
        weights = models.ResNet50_Weights.DEFAULT if pretrained else None
        model = models.resnet50(weights=weights)
        model.fc = nn.Linear(model.fc.in_features, num_classes)
        return model

    elif model_name == "vgg19":
        weights = models.VGG19_Weights.DEFAULT if pretrained else None
        model = models.vgg19(weights=weights)
        model.classifier[6] = nn.Linear(model.classifier[6].in_features, num_classes)
        return model

    elif model_name == "densenet121":
        weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
        model = models.densenet121(weights=weights)
        model.classifier = nn.Linear(model.classifier.in_features, num_classes)
        return model

    elif model_name == "efficientnet_b0":
        weights = models.EfficientNet_B0_Weights.DEFAULT if pretrained else None
        model = models.efficientnet_b0(weights=weights)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
        return model

    else:
        raise ValueError(f"Unknown model name: {model_name}")

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    return running_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0

    all_labels = []
    all_preds = []

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = criterion(outputs, labels)

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())

    return running_loss / total, correct / total, np.array(all_labels), np.array(all_preds)

In [ ]:
def compute_multiclass_metrics(y_true, y_pred, num_classes=100):
    cm = confusion_matrix(y_true, y_pred, labels=list(range(num_classes)))

    TP = 0
    TN = 0
    FP = 0
    FN = 0

    for i in range(num_classes):
        tp = cm[i, i]
        fn = cm[i, :].sum() - tp
        fp = cm[:, i].sum() - tp
        tn = cm.sum() - (tp + fn + fp)

        TP += tp
        TN += tn
        FP += fp
        FN += fn

    accuracy = (y_true == y_pred).mean()
    f1 = f1_score(y_true, y_pred, average="macro")

    return {
        "Accuracy": accuracy,
        "TP": int(TP),
        "TN": int(TN),
        "FP": int(FP),
        "FN": int(FN),
        "F1-score": f1
    }

In [ ]:
def train_model(model, model_name, train_loader, val_loader, epochs, lr, weight_decay):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="min", patience=3, factor=0.5)

    history = {
        "train_loss": [],
        "val_loss": [],
        "train_acc": [],
        "val_acc": []
    }

    best_model_wts = copy.deepcopy(model.state_dict())
    best_val_acc = 0.0
    patience = 7
    patience_counter = 0

    for epoch in range(epochs):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc, _, _ = evaluate(model, val_loader, criterion)

        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"{model_name} | Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f}")
        print(f"Val Loss:   {val_loss:.4f}, Val Acc:   {val_acc:.4f}")
        print("-" * 40)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_wts = copy.deepcopy(model.state_dict())
            patience_counter = 0
        else:
            patience_counter += 1

        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

    model.load_state_dict(best_model_wts)
    return model, history

In [ ]:
def plot_history(history, model_name):
    # Loss plot
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_loss"], label="Train Loss")
    plt.plot(history["val_loss"], label="Validation Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title(f"{model_name} Loss Curve")
    plt.legend()
    plt.show()

    # Accuracy plot
    plt.figure(figsize=(8, 5))
    plt.plot(history["train_acc"], label="Train Accuracy")
    plt.plot(history["val_acc"], label="Validation Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.title(f"{model_name} Accuracy Curve")
    plt.legend()
    plt.show()

In [ ]:
def get_first_conv_weights(model):
    for module in model.modules():
        if isinstance(module, nn.Conv2d):
            return module.weight.detach().cpu().numpy()
    raise ValueError("No Conv2d layer found.")

In [ ]:
def plot_filters(weights, title, num_filters=2):
    num_filters = min(num_filters, weights.shape[0])

    fig, axes = plt.subplots(1, num_filters, figsize=(4 * num_filters, 4))
    if num_filters == 1:
        axes = [axes]

    for i in range(num_filters):
        w = weights[i]
        w = np.transpose(w, (1, 2, 0))

        w_min, w_max = w.min(), w.max()
        if w_max > w_min:
            w = (w - w_min) / (w_max - w_min)

        axes[i].imshow(w)
        axes[i].set_title(f"Filter {i}")
        axes[i].axis("off")

    plt.suptitle(title)
    plt.tight_layout()
    plt.show()

### Train Custom CNN

In [ ]:
model_name = "CustomCNN"
train_loader, val_loader, test_loader = get_dataloaders(model_name, batch_size=128)

model = get_model(model_name, num_classes=100, pretrained=False).to(device)

weights_before = get_first_conv_weights(model)

model, history = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=25,
    lr=1e-3,
    weight_decay=1e-4
)

plot_history(history, model_name)

weights_after = get_first_conv_weights(model)
plot_filters(weights_before, "CustomCNN Filters Before Training", num_filters=2)
plot_filters(weights_after, "CustomCNN Filters After Training", num_filters=2)

#### Pretrained models

In [ ]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

import sys
print(sys.executable)
print(sys.version)

import torch
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())

In [ ]:
import sys, torch
print(sys.executable)
print(torch.__version__)
print(torch.version.cuda)
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ResNet50
model_name = "ResNet50"
train_loader, val_loader, test_loader = get_dataloaders(model_name, batch_size=64)

model = get_model(model_name, num_classes=100, pretrained=True).to(device)

model, history_resnet = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=12,
    lr=1e-4,
    weight_decay=1e-4
)

plot_history(history_resnet, model_name)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
metrics_resnet = compute_multiclass_metrics(y_true, y_pred, num_classes=100)
print(metrics_resnet)

In [ ]:
# VGG19
model_name = "VGG19"
train_loader, val_loader, test_loader = get_dataloaders(model_name, batch_size=32)

model = get_model(model_name, num_classes=100, pretrained=True).to(device)

model, history_vgg = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=12,
    lr=1e-4,
    weight_decay=1e-4
)

plot_history(history_vgg, model_name)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
metrics_vgg = compute_multiclass_metrics(y_true, y_pred, num_classes=100)
print(metrics_vgg)

In [ ]:
# DenseNet121
model_name = "DenseNet121"
train_loader, val_loader, test_loader = get_dataloaders(model_name, batch_size=64)

model = get_model(model_name, num_classes=100, pretrained=True).to(device)

model, history_dense = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=12,
    lr=1e-4,
    weight_decay=1e-4
)

plot_history(history_dense, model_name)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
metrics_dense = compute_multiclass_metrics(y_true, y_pred, num_classes=100)
print(metrics_dense)

In [ ]:
#EfficientNet_B0
model_name = "EfficientNet_B0"
train_loader, val_loader, test_loader = get_dataloaders(model_name, batch_size=64)

model = get_model(model_name, num_classes=100, pretrained=True).to(device)

model, history_eff = train_model(
    model=model,
    model_name=model_name,
    train_loader=train_loader,
    val_loader=val_loader,
    epochs=12,
    lr=1e-4,
    weight_decay=1e-4
)

plot_history(history_eff, model_name)

criterion = nn.CrossEntropyLoss()
test_loss, test_acc, y_true, y_pred = evaluate(model, test_loader, criterion)
metrics_eff = compute_multiclass_metrics(y_true, y_pred, num_classes=100)
print(metrics_eff)

#### Plot Results

In [ ]:
results = []

metrics_custom["Model"] = "CustomCNN"
metrics_resnet["Model"] = "ResNet50"
metrics_vgg["Model"] = "VGG19"
metrics_dense["Model"] = "DenseNet121"
metrics_eff["Model"] = "EfficientNet_B0"

results.append(metrics_custom)
results.append(metrics_resnet)
results.append(metrics_vgg)
results.append(metrics_dense)
results.append(metrics_eff)

df_results = pd.DataFrame(results)
df_results = df_results[["Model", "Accuracy", "TP", "TN", "FP", "FN", "F1-score"]]
df_results